# Public Key Cryptography

Learn asymmetric encryption using RSA public/private key pairs.

In [1]:
from pathlib import Path

from Crypto.Cipher import PKCS1_OAEP
from Crypto.PublicKey import RSA

# Search for key files starting from current directory and moving up
def find_project_root():
    current = Path.cwd()
    # Try current directory first
    if (current / "my_keypair").exists():
        return current
    # Try parent directory
    if (current.parent / "my_keypair").exists():
        return current.parent
    # Try two levels up (for nested structures)
    if (current.parent.parent / "my_keypair").exists():
        return current.parent.parent
    return current  # Default to current if not found

PROJECT_FOLDER = find_project_root()
print(f"Looking for keypairs in folder: {PROJECT_FOLDER}")

PRIVATE_KEY_FILE = PROJECT_FOLDER / "my_keypair"
PUBLIC_KEY_FILE = PROJECT_FOLDER / "my_keypair.pub"

assert PRIVATE_KEY_FILE.exists(), f"Private key not found at {PRIVATE_KEY_FILE}. CWD: {Path.cwd()}"
assert PUBLIC_KEY_FILE.exists(), f"Public key not found at {PUBLIC_KEY_FILE}"

print(f"✓ Found private key: {PRIVATE_KEY_FILE}")
print(f"✓ Found public key: {PUBLIC_KEY_FILE}")


Looking for keypairs in folder: /Users/lamanhuseynova/ECBS-5147-Data-Engineering-Cloud-Computing-Managed-AI-Services
✓ Found private key: /Users/lamanhuseynova/ECBS-5147-Data-Engineering-Cloud-Computing-Managed-AI-Services/my_keypair
✓ Found public key: /Users/lamanhuseynova/ECBS-5147-Data-Engineering-Cloud-Computing-Managed-AI-Services/my_keypair.pub


Load the private key from file
The private key must be kept secret and secure
Extract the public key from our private key
The public key can be freely shared with anyone


In [2]:
with open(PRIVATE_KEY_FILE, "r", encoding="utf8") as key_file:
    private_key = RSA.import_key(key_file.read())

public_key = private_key.publickey()
print(f"Public key:\n{public_key.export_key().decode('utf-8')}")


Public key:
-----BEGIN PUBLIC KEY-----
MIIBojANBgkqhkiG9w0BAQEFAAOCAY8AMIIBigKCAYEA6IdooIeJTHGHlXmk9+By
/hxaWBt+ljZWhcBT8AQirMSYvKxBaGs3qz1iNHijdxj+etak8KkHnZuK7PnaBsX+
kps5dYQ9JXM6mPsWFlQFXDeDZ24lpvjAObo4HEuTp8BaBUBvjSa2L6sD9dWdArt/
FnbWC1OOD6Xfo+GsBfSvwDTMXjRCX2b2KBGHZL/y8AUeAykGw2QVX4scxm3XBwdT
EANsJJygINT3ijFxcN2OSM2VJ46uacVPYF9eRl6NLc5gYoQY2tXPaPYzaw2/ZA0F
j0cZd8iVs2FLP3Wd0TVFtmMbtxNIMGsJAZ+6DUYl9j3lSRLmk2GXRkRUIk+38XcI
wlmJwvNCuAzVml8Xt33ngW1M+/C/gepawZTv4twXA4PTmwXHUlHQQad+YzuZF5dZ
5suSKhZ6PPNS7uxVNQewpVpmUS48swKuIK+Y/yHTEay7ItzWDZL1CjAQXEROlli6
s0HcvfTTJ6HVotUCC/mT1hkjcacxF5PPjdrp1ZwiJphNAgMBAAE=
-----END PUBLIC KEY-----


Message to be encrypted - must be converted to bytes
Create a cipher object using the public key for encryption
Encrypt our message - only someone with the private key can decrypt it
Save the encrypted message to a file


In [3]:
short_secret_message = "My Secret Message".encode("utf-8")

public_key_cipher = PKCS1_OAEP.new(public_key)

encrypted_message = public_key_cipher.encrypt(short_secret_message)
print("Encrypted message:")
print(encrypted_message.hex())

ENCRYPTED_MESSAGE_FILE = PROJECT_FOLDER / "encrypted_message.bin"
with open(ENCRYPTED_MESSAGE_FILE, "wb") as f:
    f.write(encrypted_message)


Encrypted message:
659ac86f9ada35a3e26d85dfabdafa1739a21701081954815eb728b09c745354961a75aa23a751adae8ca1e9e1e61f184ea08ff6701905df2c7f1e30d860542e57dae8df623f6cfb359e60e995b8e724fac7ef8b2591855e92da70614d00851d8a3d83704c84a197820dd26c97823747a711b11968eb5c5f2260c2d56977f51f55bd063586157d12f230db3b57939b45b70c6e2981c925acca53ef06078bb2072df5ec77a2f0fb9cb5d601f95c6f981ce25d16e426ec38bfcd80c2770c56f73890edb2031fd3322012c484efbdf9fdf7b6df7a538751bbc4e13cc071b56c645f6c2e96346dbdaa968d524909dc3e2abdde7e00d8dda7360e116e835a1594d53c3e9b87054559a0b6c2208e24bbecc46081fd0ba976d427c1a2ba8f4daf662d233c432befc8dbc5157d602be062c89cafcb805edced7d684ca2ebb20b23e99cbae9a651e247d9c1d63cd8009c781ccf37d56209ea820aadea83d91cd1a3e6e718974e6c58459716bb69a80fc965d3524492bf6083631b30f1bc82e0d9dec5f48c


Read the encrypted message from file
Create a cipher object using the private key for decryption
Decrypt the message using the private key
This demonstrates asymmetric encryption where:
1. Messages are encrypted with a public key and can only be decrypted with the matching private key
2. The public key can be freely shared
3. The private key must be kept secret
4. This is more secure than symmetric encryption as the decryption key (private key) is never shared


In [4]:
with open(ENCRYPTED_MESSAGE_FILE, "rb") as f:
    encrypted_message_from_file = f.read()

private_key_cipher = PKCS1_OAEP.new(private_key)

decrypted_message = private_key_cipher.decrypt(encrypted_message_from_file)
print(f"Decrypted message: {decrypted_message.decode('utf-8')}")



Decrypted message: My Secret Message
